# Explanation Service with an Audit Trail — Results Notebook

**Project:** Big Data Systems and Responsible AI mini project  
**Persons:** 1 (Storage Design), 2 (Query Interface & Reproducibility), 3 (Compliance & Audit Trail)  
**Backend:** DuckDB/Parquet by default (Apache Iceberg via `EXPLANATION_REPO_BACKEND=iceberg`)

This notebook walks through the complete project end-to-end: storage design, query interface, reproducibility replay, benchmarks, and compliance reporting.

## 0. Setup

Install dependencies and confirm the environment. If the service is not already running, start it in another terminal:

```bash
EXPLANATION_REPO_BACKEND=duckdb uvicorn person2_query_interface.app.service:app
```

In [ ]:
import sys
from pathlib import Path

print("Python:", sys.version)
repo = Path.cwd()
checks = [
    "person2_query_interface/app/service.py",
    "person2_query_interface/sql/queries.sql",
    "person2_query_interface/data/monthly_batches/2026-10/batch_manifest.json",
    "iceberg_schema.sql",
    "MODEL_CARD.md",
    "DATA_CARD.md",
    "reports/technical_report.md",
]
for p in checks:
    print(("OK  " if (repo / p).exists() else "MISS"), p)

## 1. Storage design (Person 1)

Review the Iceberg schema and a sample synthetic audit record.

In [ ]:
from pathlib import Path
import json as _j
print(Path("iceberg_schema.sql").read_text()[:1500])
print("...")
print("
--- synthetic record (first prediction event) ---")
rec = _j.loads(Path("synthetic_audit_records.json").read_text())
print(_j.dumps(rec["prediction_events"][0], indent=2))

## 2. Start the service and query `/health`

If the service is not running, this cell starts it in the background.

In [ ]:
import os, subprocess, time, urllib.request, json as _j, sys

BASE = "http://127.0.0.1:8000"

def healthy():
    try:
        with urllib.request.urlopen(BASE + "/health", timeout=2) as r:
            return _j.loads(r.read())
    except Exception:
        return None

h = healthy()
if h is None:
    print("Starting service ...")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn",
         "person2_query_interface.app.service:app",
         "--host", "127.0.0.1", "--port", "8000"],
        env={**os.environ, "EXPLANATION_REPO_BACKEND": "duckdb"},
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        time.sleep(1)
        h = healthy()
        if h:
            break
    print("Service PID:", proc.pid)
print("Health:", _j.dumps(h, indent=2))

## 3. `GET /explain` — "Why was this decision made?"

Look up the most recent decision for a synthetic `request_trace_id` within a time window. The response includes observability fields (`latency_ms`, `bytes_scanned`, `within_slo`) against the 5-second SLO.

In [ ]:
)
import urllib.request, json as _j
from person2_query_interface.app.repository import get_repository

repo = get_repository()
ids = repo.list_prediction_ids()
pred_id = ids[0]
bundle = repo.get_decision_bundle(pred_id)
trace = bundle.prediction.request_trace_id
ts = bundle.prediction.decision_ts
url = f"{BASE}/explain?request_trace_id={trace}&from_ts={ts}&to_ts={ts}"
with urllib.request.urlopen(url) as r:
    explain = _j.loads(r.read())

print("resolved_prediction_id:", explain["query"]["resolved_prediction_id"])
print("demo_decision:", explain["decision"]["prediction"]["demo_decision"])
print("risk_score:", explain["decision"]["prediction"]["risk_score"])
print("model_version:", explain["decision"]["prediction"]["model_version"])
print("explainer:", explain["decision"]["explanation"]["explainer_name"],
      explain["decision"]["explanation"]["explainer_version"])
print("observability:", explain["observability"])

## 4. `GET /decision/{id}` — full audit bundle

Return the complete audit answer: prediction event joined to exactly one input snapshot, one explanation record, and one model build (Q2).

In [ ]:
)
with urllib.request.urlopen(f"{BASE}/decision/{pred_id}") as r:
    decision = _j.loads(r.read())
d = decision["decision"]
print("prediction_id:", d["prediction"]["prediction_id"])
print("decision_ts:", d["prediction"]["decision_ts"])
print("demo_decision:", d["prediction"]["demo_decision"], "| risk_score:", d["prediction"]["risk_score"])
print("request_trace_id:", d["prediction"]["request_trace_id"])
print("input_snapshot_id:", d["input_snapshot"]["input_snapshot_id"])
print("income_total:", d["input_snapshot"]["income_total"], "| credit_amount:", d["input_snapshot"]["credit_amount"])
print("explanation_id:", d["explanation"]["explanation_id"])
print("explainer:", d["explanation"]["explainer_name"], d["explanation"]["explainer_version"])
print("reason_codes:", d["explanation"]["reason_codes"])
print("rendered_summary:", d["explanation"]["rendered_summary"])
print("model_version:", d["model"]["model_version"], "| threshold_policy:", d["model"]["threshold_policy_version"])
print("artifact_sha256:", d["model"]["artifact_sha256"])

## 5. `GET /replay/{id}` — reproducibility replay

Re-run the pinned model on the stored input and validate score (<=1e-6), exact decision, reason codes, contribution ranking, and input/explanation hashes against the audit trail.

In [ ]:
)
with urllib.request.urlopen(f"{BASE}/replay/{pred_id}") as r:
    replay = _j.loads(r.read())
for k in ["reproducible","score_match","decision_match","explanation_match",
          "input_hash_verified","explanation_hash_verified"]:
    print(k, ":", replay[k])
print("original_score:", replay["original_score"], "| replayed_score:", replay["replayed_score"])
print("score_diff:", replay["score_diff"], "| tolerance:", replay["tolerance"])
print("original_decision:", replay["original_decision"], "| replayed_decision:", replay["replayed_decision"])
print("model_version:", replay["model_version"], "| explainer:", replay["explainer_name"], replay["explainer_version"])

## 6. `GET /decision/{id}/at/{snapshot}` — point-in-time read

Time-travel equivalent: retrieve the bundle as it existed at a given snapshot (Iceberg `VERSION AS OF`; DuckDB backend emulates via immutable monthly batch).

In [ ]:
)
from pathlib import Path
import json as _j2
manifest_path = Path("person2_query_interface/data/monthly_batches/2026-10/batch_manifest.json")
snap = None
if manifest_path.exists():
    manifest = _j2.loads(manifest_path.read_text())
    snap = manifest["snapshot_id"]
print("snapshot_id:", snap)
if snap:
    with urllib.request.urlopen(f"{BASE}/decision/{pred_id}/at/{snap}") as r:
        pit = _j.loads(r.read())
    print("point-in-time prediction_id:", pit["decision"]["prediction"]["prediction_id"])
else:
    print("no manifest snapshot available; skipping point-in-time demo")

## 7. `GET /consistency` — decision-consistency report

Compliance view: did the same inputs always produce the same decision across model versions and time?

In [ ]:
)
with urllib.request.urlopen(f"{BASE}/consistency") as r:
    consistency = _j.loads(r.read())
print(_j.dumps(consistency, indent=2)[:2000])

## 8. Performance benchmarks

Run p50/p95 latency benchmarks across 1k / 5k / 10k records against the 5-second SLO.

In [ ]:
)
from pathlib import Path
from person2_query_interface.benchmarks.run_benchmarks import main as bench_main
bench_main()
import json as _j
res = _j.loads(Path("person2_query_interface/benchmarks/results/latest.json").read_text())
for s in res["scales"]:
    print(f"{s['record_count']:>6} rec | p50={s['latency_ms']['p50']:>7} ms | p95={s['latency_ms']['p95']:>7} ms | "
          f"bytes={s['bytes_scanned']['p50']/1024:>8.1f} KB | within_slo={s['within_slo']}")
print("SLO seconds:", res["slo_seconds"], "| backend:", res["backend"])

## 9. Person 3 compliance report

Role-based access control, hash-chained audit logging, consistency, reproducibility, completeness, and integrity checks.

In [ ]:
)
import sys
sys.path.insert(0, "person3_compliance_audit")
import unittest
loader = unittest.TestLoader()
suite = loader.discover("person3_compliance_audit/tests")
runner = unittest.TextTestRunner(verbosity=1, stream=sys.stdout)
result = runner.run(suite)
print("
Tests run:", result.testsRun, "| Failures:", len(result.failures), "| Errors:", len(result.errors))
print("Overall:", "PASS" if result.wasSuccessful() else "FAIL")

## 10. Reproducible pipeline

The entire project can be reproduced with the scripted pipeline:

```bash
./scripts/pipeline.sh all
# install -> generate -> test -> serve -> demo -> benchmark
```

Containerized alternative:

```bash
docker compose up --build        # DuckDB/Parquet backend on port 8000
docker compose up --build iceberg  # Apache Iceberg backend on port 8001
python init_iceberg.py           # one-time Iceberg warehouse init + load
```

## 11. Conclusion

The service answers "why was this decision made?" with a full, immutable audit bundle; reproduces any historical decision exactly (score within 1e-6, decision exact, hashes verified); meets the 5-second SLO with wide margin; and produces a `COMPLIANT` regulatory report under role-based access control with hash-chained audit logging — while never storing direct borrower identifiers.